# 02 — Classical Linear Baseline: TF-IDF + Logistic Regression (Run A & Run B)

**Purpose:** Evaluate classical sparse surface-feature baselines to quantify how much of the joint task can be solved by character and word $n$-grams without pretrained multilingual transformers.

- **Primary Variant (`tfidf_char_lr`):**
  - `TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=2, sublinear_tf=True)`
  - Two independent `LogisticRegression(class_weight='balanced', max_iter=2000)` classifiers (one for 4-class sentiment, one for 6-class intent).
- **Secondary Variant (`tfidf_word_lr`):**
  - `TfidfVectorizer(analyzer='word', ngram_range=(1,2), min_df=2, sublinear_tf=True)`
  - Same independent `LogisticRegression(class_weight='balanced', max_iter=2000)` setup.
- **Hyperparameter Tuning Protocol:**
  - Tune only $C \in \{0.1, 1, 10\}$ on the **validation** split (`val_df`), selecting $C^*$ by mean validation Macro-F1 across the two tasks: $(\text{F1}_{\text{sent}} + \text{F1}_{\text{intent}}) / 2$.
  - Evaluate the selected model on the 300-row human gold test set **once**.
- **Compute:** Deterministic, CPU only.

## 1. Environment & Single Config Cell

In [ ]:
# ── Colab / Repo-Root Setup ───────────────────────────────────────────────────
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("eval_utils.py"):
    if not os.path.exists("/content/Sinhala-joint-sentiment-and-intent"):
        subprocess.run(["git", "clone", "https://github.com/ThiwankaChanditha/Sinhala-joint-sentiment-and-intent.git", "/content/Sinhala-joint-sentiment-and-intent"], check=True)
    os.chdir("/content/Sinhala-joint-sentiment-and-intent")
elif not os.path.exists("eval_utils.py"):
    for _cand in ["..", "../..", "/content/Sinhala-joint-sentiment-and-intent"]:
        if os.path.exists(os.path.join(_cand, "eval_utils.py")):
            os.chdir(_cand)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import os
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from eval_utils import (
    SENT_CLASSES,
    INTENT_CLASSES,
    NS,
    NI,
    log_environment,
    set_all_seeds,
    load_splits,
    evaluate_and_save,
)

# ── Single Config Cell ────────────────────────────────────────────────────────
SEED          = 42
C_GRID        = [0.1, 1.0, 10.0]
RUNS_TO_EXEC  = ["runA", "runB"]

SILVER_CSV    = None   # None = auto-resolve sinhala_silver_labels.csv
GOLD_CSV      = None   # None = auto-resolve annotation_with_sentiment_intent.csv
RESULTS_DIR   = "results"
PREDS_DIR     = "preds"

VARIANTS = {
    "char": {
        "notebook_tag": "02_tfidf_lr_baseline_char",
        "model_tag": "tfidf_char_lr",
        "display_name": "TF-IDF + LR (char)",
        "vec_kwargs": dict(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
    },
    "word": {
        "notebook_tag": "02_tfidf_lr_baseline_word",
        "model_tag": "tfidf_word_lr",
        "display_name": "TF-IDF + LR (word)",
        "vec_kwargs": dict(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    },
}

set_all_seeds(SEED)
env_info = log_environment()

## 2. Validation Tuning ($C \in \{0.1, 1, 10\}$) & Single Gold Test Evaluation

In [ ]:
val_tuning_logs = []
test_summary_rows = []

for run in RUNS_TO_EXEC:
    print(f"\n{'='*75}\nDataset Regime: {run}\n{'='*75}")
    train_df, val_df, test_df = load_splits(run=run, silver_csv=SILVER_CSV, gold_csv=GOLD_CSV)

    y_tr_s, y_tr_i = train_df["sent_id"].values, train_df["intent_id"].values
    y_va_s, y_va_i = val_df["sent_id"].values,   val_df["intent_id"].values

    for v_key, cfg in VARIANTS.items():
        vec = TfidfVectorizer(**cfg["vec_kwargs"])
        X_tr = vec.fit_transform(train_df["text"].tolist())
        X_va = vec.transform(val_df["text"].tolist())
        X_te = vec.transform(test_df["text"].tolist())

        best_c = None
        best_val_mean_f1 = -1.0
        best_models = None
        tuning_history = []

        for c_val in C_GRID:
            clf_s = LogisticRegression(
                C=c_val, class_weight="balanced", max_iter=2000, random_state=SEED
            )
            clf_i = LogisticRegression(
                C=c_val, class_weight="balanced", max_iter=2000, random_state=SEED
            )
            clf_s.fit(X_tr, y_tr_s)
            clf_i.fit(X_tr, y_tr_i)

            val_pred_s = clf_s.predict(X_va)
            val_pred_i = clf_i.predict(X_va)

            val_f1_s = float(f1_score(y_va_s, val_pred_s, labels=list(range(NS)), average="macro", zero_division=0))
            val_f1_i = float(f1_score(y_va_i, val_pred_i, labels=list(range(NI)), average="macro", zero_division=0))
            val_mean_f1 = 0.5 * (val_f1_s + val_f1_i)

            tuning_history.append({
                "C": c_val,
                "val_sent_f1": round(val_f1_s, 4),
                "val_intent_f1": round(val_f1_i, 4),
                "val_mean_f1": round(val_mean_f1, 4),
            })
            val_tuning_logs.append({
                "run": run,
                "variant": cfg["display_name"],
                "vocab_size": len(vec.vocabulary_),
                **tuning_history[-1],
            })

            if val_mean_f1 > best_val_mean_f1:
                best_val_mean_f1 = val_mean_f1
                best_c = c_val
                best_models = (clf_s, clf_i)

        print(f"[{cfg['display_name']} | {run}] Vocab={len(vec.vocabulary_):,} | Best C={best_c} (val mean macro-F1={best_val_mean_f1:.4f})")

        # Evaluate once on the 300-row gold test set using the val-selected C
        best_clf_s, best_clf_i = best_models
        te_probs_s = best_clf_s.predict_proba(X_te)
        te_probs_i = best_clf_i.predict_proba(X_te)
        te_pred_s  = te_probs_s.argmax(axis=1)
        te_pred_i  = te_probs_i.argmax(axis=1)

        payload = evaluate_and_save(
            notebook_name=cfg["notebook_tag"],
            model_name=cfg["model_tag"],
            run=run,
            seed=SEED,
            test_df=test_df,
            pred_sent_id=te_pred_s,
            pred_intent_id=te_pred_i,
            sent_probs=te_probs_s,
            intent_probs=te_probs_i,
            extra_meta={
                "vectorizer": cfg["vec_kwargs"],
                "vocab_size": len(vec.vocabulary_),
                "best_C": best_c,
                "best_val_mean_f1": best_val_mean_f1,
                "val_tuning_grid": tuning_history,
            },
            results_dir=RESULTS_DIR,
            preds_dir=PREDS_DIR,
        )

        if v_key == "char":
            alias_path = os.path.join(RESULTS_DIR, f"02_tfidf_lr_baseline_{run}_{SEED}.json")
            with open(alias_path, "w", encoding="utf-8") as f:
                json.dump(payload, f, indent=2)

        test_summary_rows.append({
            "Run": run,
            "Model": cfg["display_name"],
            "Best C (Val)": best_c,
            "Val Mean F1": round(best_val_mean_f1, 3),
            "Sent Acc": round(payload["sent_acc"], 3),
            "Sent F1 (3-cls)": round(payload["sent_f1_3cls"], 3),
            "Sent F1 (4-cls)": round(payload["sent_f1_4cls"], 3),
            "Intent Acc": round(payload["intent_acc"], 3),
            "Intent F1": round(payload["intent_f1"], 3),
            "Joint Acc": round(payload["joint_acc"], 3),
        })

print("\n=== Validation Hyperparameter Sweep (C in {0.1, 1, 10}) ===")
display(pd.DataFrame(val_tuning_logs))

print("\n=== Gold Test Set Results (300 Human-Annotated Rows) ===")
display(pd.DataFrame(test_summary_rows))